# 音声認識の結果から固有表現を抽出する

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/19_ner/asr_ner.ipynb)

NLP クックブック 19-13 ／ 固有表現抽出

会議の録音やコールセンターの通話のような音声から、人名・地名・組織名を取り出したい。音声認識の誤りが固有表現抽出にどう影響するかも確かめる。

In [1]:
!pip install -q gTTS ginza ja-ginza jiwer
!apt-get install -y -q ffmpeg > /dev/null 2>&1

In [2]:
from gtts import gTTS

# 手元に音声がないので、ここでは合成音声で試す (実際には録音した音声ファイルを使う)
texts = [
    "山田太郎さんは来週の月曜日に大阪の本社で開かれる会議に出席します。",
    "京都大学の研究チームは、理化学研究所と共同で新しい実験を行いました。",
    "ソニーとホンダは、電気自動車を開発する新会社を東京に設立すると発表した。",
    "総理大臣は明日、ワシントンでアメリカの大統領と会談する予定です。",
    "鈴木花子選手は、パリで行われた大会で金メダルを獲得しました。",
    "北海道の函館市では、昨日から大雪が続いており、JR 北海道は一部の列車を運休しています。",
]
for i, t in enumerate(texts):
    gTTS(t, lang="ja").save(f"audio{i}.mp3")
!ls -la audio*.mp3 | head -n 3

-rw-r--r-- 1 root root 61248 Nov 15 12:17 audio0.mp3
-rw-r--r-- 1 root root 70272 Nov 15 12:17 audio1.mp3
-rw-r--r-- 1 root root 69312 Nov 15 12:17 audio2.mp3


In [3]:
import torch
from transformers import pipeline

asr = pipeline("automatic-speech-recognition", model="openai/whisper-small", generate_kwargs={"language": "japanese", "task": "transcribe"})
transcripts = [asr(f"audio{i}.mp3")["text"].strip() for i in range(len(texts))]
for ref, hyp in zip(texts, transcripts):
    print(f"正解: {ref}\n認識: {hyp}")

正解: 山田太郎さんは来週の月曜日に大阪の本社で開かれる会議に出席します。
認識: 山田太郎さんは来週の月曜日に大阪の本社で開かれる会議に出席します
正解: 京都大学の研究チームは、理化学研究所と共同で新しい実験を行いました。
認識: 京都大学の研究チームは、理科学研究所と共同で新しい実験を行いました。
正解: ソニーとホンダは、電気自動車を開発する新会社を東京に設立すると発表した。
認識: ソニーと本田は、電気自動車を開発する新会社を東京に設立すると発表した。
正解: 総理大臣は明日、ワシントンでアメリカの大統領と会談する予定です。
認識: 総理大臣は明日、ワシントンでアメリカの大統領と会談する予定です。
正解: 鈴木花子選手は、パリで行われた大会で金メダルを獲得しました。
認識: 鈴木花子選手は、パリで行われた大会で金メダルを獲得しました。
正解: 北海道の函館市では、昨日から大雪が続いており、JR 北海道は一部の列車を運休しています。
認識: 北海道の箱立て市では、昨日から大雪が続いており、JR北海道は一部の列車を運休しています。


In [4]:
import jiwer
import spacy

nlp = spacy.load("ja_ginza")
normalize = lambda s: s.replace(" ", "").replace("、", "").replace("。", "")
print("文字誤り率 (CER):", round(jiwer.cer([normalize(t) for t in texts], [normalize(t) for t in transcripts]), 3))

def entities(text):
    return {(e.text, e.label_) for e in nlp(text).ents}

tp = fp = fn = 0
for ref, hyp in zip(texts, transcripts):
    gold, pred = entities(ref), entities(hyp)          # 正解の書き起こしから取り出した固有表現を「正解」と見なす
    tp += len(gold & pred); fp += len(pred - gold); fn += len(gold - pred)
    if gold != pred:
        print(f"書き起こしから: {sorted(gold - pred)}\n認識結果から  : {sorted(pred - gold)}")
p, r = tp / (tp + fp), tp / (tp + fn)
print(f"固有表現の一致: 適合率 {p:.2f} / 再現率 {r:.2f} / F1 {2 * p * r / (p + r):.2f}")

文字誤り率 (CER): 0.036
書き起こしから: [('理化学研究所', 'Position_Vocation')]
認識結果から  : [('理科学研究所', 'Facility_Other')]
書き起こしから: [('ホンダ', 'Company')]
認識結果から  : [('本田', 'Person')]
書き起こしから: [('JR 北海道', 'Station'), ('函館市', 'City')]
認識結果から  : [('JR北海道', 'Product_Other')]
固有表現の一致: 適合率 0.83 / 再現率 0.79 / F1 0.81


In [5]:
# 表記のゆれを吸収して比べる: 読み (カタカナ) が同じなら一致と見なす
def readings(text):
    doc = nlp(text)
    return {("".join(t.morph.get("Reading")[0] if t.morph.get("Reading") else t.text for t in e), e.label_) for e in doc.ents}

tp = fp = fn = 0
for ref, hyp in zip(texts, transcripts):
    gold, pred = readings(ref), readings(hyp)
    tp += len(gold & pred); fp += len(pred - gold); fn += len(gold - pred)
p, r = tp / (tp + fp), tp / (tp + fn)
print(f"読みで比べた場合: 適合率 {p:.2f} / 再現率 {r:.2f} / F1 {2 * p * r / (p + r):.2f}")

読みで比べた場合: 適合率 0.83 / 再現率 0.79 / F1 0.81


- 音声認識の結果には、句読点がない（またはモデルが推定したもの）、同音異義語の誤り（人名の漢字が違う）、数字や英字の表記のゆれ、といった書き言葉との違いがある。書き言葉で学習した固有表現抽出器は、その分だけ精度が落ちる。
- 固有名詞は音声認識が最も間違えやすい部分。特に人名は、音が合っていても漢字が違うことが多い。読みで照合する、候補の辞書と突き合わせる、といった後処理が有効。
- 読みで照合しても、この例では結果が変わらなかった。「ホンダ」が「本田」になると、表記だけでなく種類の判定まで（会社 → 人名）変わってしまうため。音声認識の誤りは、固有表現の範囲と種類の両方に波及する。
- 全体の文字誤り率が低くても、固有表現に限ると誤りが多いことがある。目的が固有表現なら、固有表現だけの誤り率を測る。
- ここで使ったのは合成音声で、実際の会話音声（雑音、言いよどみ、複数話者）よりはるかに認識しやすい。実データでの性能は、必ず実データで測る。
- 音声認識と固有表現抽出を別々に行う方法（ここでの方法）のほかに、音声から直接固有表現を出すモデルや、認識結果の誤りを LLM で訂正してから抽出する方法がある。